# Lightweight CNN Experiment

This notebook analyzes the lightweight CNN based on depthwise separable convolutions.

The research constraint is:

> Fewer than 100,000 trainable parameters.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch

from src.models.lightweight_cnn import LightweightCNN
from src.models.model_utils import count_parameters, model_size_mb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LightweightCNN().to(device)

parameters = count_parameters(model)

print("Device:", device)
print("Trainable parameters:", parameters)
print("Model size:", model_size_mb(model), "MB")
print("Parameter constraint:", parameters < 100000)


Device: cuda
Trainable parameters: 3989
Model size: 0.016490936279296875 MB
Parameter constraint: True


In [2]:
from src.data.dataset import get_dataloaders

_, _, test_loader = get_dataloaders(batch_size=128)

model_path = ROOT / "models" / "lightweight_cnn" / "best_model.pth"

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 97.22%


## Lightweight CNN Result

The trained lightweight CNN achieved approximately **97.23% test accuracy**.

It contains only **3,989 trainable parameters**, which is far below the 100,000-parameter research constraint.
